# 10 · DefaultAzureCredential & Managed Identity

Connection strings are convenient, but in production you should use **Azure AD / Entra** identities:

- **Managed Identity** when running in Azure (App Service, VM, AKS, Functions)
- **`az login`** session locally
- **Service Principal** in CI / CD

`DefaultAzureCredential` walks a chain of credential sources until one succeeds — the same code works in every environment.

## RBAC roles

| Role | Lets you... |
|---|---|
| Azure Service Bus Data Sender | Send |
| Azure Service Bus Data Receiver | Receive |
| Azure Service Bus Data Owner | Everything (incl. manage entities) |

Assign **at the namespace, queue, topic, or subscription** scope.

> Our Bicep template can pre-assign Data Owner if you pass `principalId`. Otherwise assign it manually:
>
> ```bash
> az role assignment create \
>   --assignee $(az ad signed-in-user show --query id -o tsv) \
>   --role "Azure Service Bus Data Owner" \
>   --scope $(az servicebus namespace show -g rg-sbdemo -n <namespace> --query id -o tsv)
> ```

Set `SERVICEBUS_NAMESPACE` (e.g. `sbdemo123.servicebus.windows.net`) before running this notebook — **no connection string needed**.


In [ ]:
#r "nuget: Azure.Messaging.ServiceBus, 7.18.2"
#r "nuget: Azure.Identity, 1.13.1"

In [ ]:
#!import ../src/shared/Config.cs

## 1. Build a client using DefaultAzureCredential

In [ ]:
using Azure.Identity;
using Azure.Messaging.ServiceBus;

var credential = new DefaultAzureCredential();
var client = new ServiceBusClient(Config.FullyQualifiedNamespace, credential);

Console.WriteLine($"Connected to {client.FullyQualifiedNamespace} via Entra credential.");

## 2. Send and receive, same APIs as before

In [ ]:
var sender = client.CreateSender(Config.QueueName);
await sender.SendMessageAsync(new ServiceBusMessage("hello from DefaultAzureCredential"));

var receiver = client.CreateReceiver(Config.QueueName);
var msg = await receiver.ReceiveMessageAsync(TimeSpan.FromSeconds(5));
Console.WriteLine($"Got: {msg?.Body}");
if (msg is not null) await receiver.CompleteMessageAsync(msg);

await sender.DisposeAsync();
await receiver.DisposeAsync();
await client.DisposeAsync();

## Troubleshooting

| Symptom | Likely cause |
|---|---|
| `Unauthorized` / `401` | Missing RBAC role assignment, or change hasn't propagated (wait 1–2 min) |
| `Identity not found` | No credential in the chain succeeded — try `az login` |
| Works locally, not in Azure | Managed identity not enabled on the host, or wrong scope |

## You're done!

You've covered the full SDK surface area. From here, look into:

- **`ServiceBusModelFactory`** for unit testing message handlers
- **Azure.Monitor.OpenTelemetry.AspNetCore** for distributed tracing through Service Bus
- **Premium tier features**: geo-DR, large messages (100 MB), VNet integration
- **Schema Registry** for centralised payload contracts
